# Caco-2 Permeability Prediction (Caco2_Wang)

This notebook retrieves the TDC Caco2_Wang dataset, preprocesses SMILES, featurizes molecules (Morgan fingerprints + descriptors), trains XGBoost and RandomForest models, evaluates on a hold-out test set (scaffold or random split), produces metrics and SHAP explanations, and saves model and predictions.

Run in a Python environment with RDKit, tdc, scikit-learn, xgboost, shap, pandas, numpy installed.

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install tdc rdkit-pypi xgboost shap scikit-learn pandas numpy joblib matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')


In [ ]:
# 1. Download dataset from TDC
data = ADME(name='Caco2_Wang')
df = data.get_data()
print('Raw rows:', len(df))
df.head()


In [ ]:
# 2. Preprocess: validate SMILES, drop invalids, drop duplicates
def validate_and_add_mol(df, smiles_col='smiles'):
    df = df.copy()
    df['mol'] = df[smiles_col].apply(lambda s: Chem.MolFromSmiles(s) if isinstance(s, str) else None)
    df = df[df['mol'].notnull()].reset_index(drop=True)
    return df
df = validate_and_add_mol(df)
df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
print('After SMILES validation and deduplication:', len(df))
df['Y'] = pd.to_numeric(df['Y'], errors='coerce')
df = df[df['Y'].notnull()].reset_index(drop=True)
print('After dropping missing labels:', len(df))
df['Y'].describe()


In [ ]:
# 3. Optional: use TDC scaffold split if available; fallback to random split
try:
    splits = data.get_split(method='scaffold')
    train_idx = splits['train']
    val_idx = splits.get('val', [])
    test_idx = splits['test']
    train_df = df.loc[train_idx].reset_index(drop=True)
    test_df  = df.loc[test_idx].reset_index(drop=True)
    print('Using TDC scaffold split: train', len(train_df), 'test', len(test_df))
except Exception as e:
    print('Scaffold split not available or failed, using random split. Error:', e)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print('Random split: train', len(train_df), 'test', len(test_df))


In [ ]:
# 4. Featurization: Morgan fingerprint + a few descriptors
def featurize_mol(mol, n_bits=2048, radius=2):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.array(list(fp.ToBitString()), dtype=np.int8) if hasattr(fp, 'ToBitString') else np.array(fp)
    # descriptors
    try:
        mw = Descriptors.MolWt(mol)
        clogp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
    except Exception:
        mw = clogp = tpsa = hbd = hba = 0.0
    descs = np.array([mw, clogp, tpsa, hbd, hba], dtype=float)
    return np.concatenate([arr.astype(float), descs])
# Build feature matrices
def build_X(df):
    feats = [featurize_mol(m) for m in df['mol']]
    return np.vstack(feats)
X_train = build_X(train_df)
y_train = train_df['Y'].values.astype(float)
X_test  = build_X(test_df)
y_test  = test_df['Y'].values.astype(float)
print('Feature shapes:', X_train.shape, X_test.shape)


In [ ]:
# 5. Baseline models: RandomForest and XGBoost. Use simple CV tuning for example.
# RandomForest grid search (simple)
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {'n_estimators':[200,500], 'max_depth':[None,10,20]}
rf_gs = GridSearchCV(rf, rf_params, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
rf_gs.fit(X_train, y_train)
print('RF best params:', rf_gs.best_params_)
best_rf = rf_gs.best_estimator_
# XGBoost grid search
xg = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)
xg_params = {'n_estimators':[100,300], 'learning_rate':[0.01,0.1], 'max_depth':[3,6]}
xg_gs = GridSearchCV(xg, xg_params, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
xg_gs.fit(X_train, y_train)
print('XGB best params:', xg_gs.best_params_)
best_xg = xg_gs.best_estimator_


In [ ]:
# 6. Evaluate on test set and save metrics and predictions
models = {'random_forest': best_rf, 'xgboost': best_xg}
out_dir = 'caco2_output'
os.makedirs(out_dir, exist_ok=True)
results = {}
for name, m in models.items():
    y_pred = m.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'rmse':rmse, 'mae':mae, 'r2':r2}
    print(f'{name}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}')
    # save model and predictions
    joblib.dump(m, os.path.join(out_dir, f'model_{name}.pkl'))
    pred_df = test_df[['smiles','Y']].copy()
    pred_df['pred'] = y_pred
    pred_df.to_csv(os.path.join(out_dir, f'predictions_{name}.csv'), index=False)
# save metrics summary
pd.DataFrame(results).T.to_csv(os.path.join(out_dir, 'metrics_summary.csv'))
print('Saved outputs to', out_dir)


In [ ]:
# 7. SHAP explanation for XGBoost (if available)
try:
    explainer = shap.TreeExplainer(best_xg)
    shap_values = explainer.shap_values(X_test)
    plt.figure(figsize=(8,6))
    shap.summary_plot(shap_values, features=X_test, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'shap_summary.png'), dpi=300)
    print('SHAP summary saved')
except Exception as e:
    print('SHAP explanation failed:', e)


## Next steps and notes
- Consider conformal prediction or ensembles for prediction intervals.
- Compute applicability domain (Tanimoto similarity to training) and add similarity column to predictions.
- For production, add input validation, logging, command-line arguments, and unit tests.
